# Animal Detection in Camera Trap Images — Final Report

**Course:** Neural Networks. Theory and Practice (2025/26)  
**Task type:** Object Detection  
**Dataset:** ENA24-detection  

**Team:**
- Kamila Korczyńska
- Kamil Tasarz

## 1. Problem Definition

Camera trap images often contain one or more animals that must be found automatically before any downstream analysis (counting, species ID, etc.). We treat this as **object detection**: given a single image, predict where animal instances are in pixel space.


### 1.1. ML Task Formulation

We formulate the problem as **object detection** on a labeled dataset

$$\mathcal{D} = \{(I_i, Y_i)\}_{i=1}^{N},$$

where $I_i \in \mathbb{R}^{H \times W \times C}$ is a camera trap image and

$$Y_i = \{(b_{ij}, \ell_{ij})\}_{j=1}^{M_i}$$

is the set of ground-truth objects in that image. Each box $b_{ij} = (x_{ij}, y_{ij}, w_{ij}, h_{ij})$ is given in COCO style (top-left corner, width, height in pixels). The label $\ell_{ij}$ is a category id in the raw annotations (species-level in ENA24). The object count $M_i \geq 1$ varies per image.

The learning goal is a detector

$$f : \mathbb{R}^{H \times W \times C} \rightarrow \{(\hat{b}_k, \hat{s}_k)\}_{k=1}^{\hat{M}}$$

that maps an unseen image to a set of predicted boxes $\hat{b}_k$ and confidence scores $\hat{s}_k$. The number of predictions $\hat{M}$ is not fixed and is determined by the model and its post-processing (e.g. thresholding and NMS).

In this project we evaluate **localization quality** (whether a predicted box overlaps a ground-truth box sufficiently). How we use—or ignore—$\ell_{ij}$ and $\hat{\ell}_k$ in training and matching is defined in Section 3.


### 1.2. Inputs, Outputs, and Objectives

| | Description |
| --- | --- |
| **Input** | One RGB camera trap image $I_i$ (variable resolution; resized or tiled as required by each model). |
| **Output** | A list of bounding boxes in image coordinates, each with an optional class id and confidence score. |
| **Primary objective** | Minimize localization error on held-out images: predicted boxes should align with annotated animal regions. |
| **Evaluation** | Class-agnostic detection metrics at a fixed match IoU (precision, recall, F1, mAP); see Section 8. |

**Practical focus:** images with a single animal ($M_i = 1$) and images with several animals ($M_i > 1$). We do not assume a fixed number of objects per image.

**Out of scope for the problem statement:** dataset download layout, train/val/test splits, and species classification—those are covered in Sections 2, 5, and 3 respectively.


## 2. Dataset: ENA24

All experiments use **[ENA24-detection](https://lila.science/datasets/ena24detection/)** — camera trap images from Eastern North America with COCO-style bounding-box annotations. We work with the **non-human subset** used in this course setup. Layout on disk: `images/` plus a single COCO JSON (`annotations.json` or equivalent metadata file).


### 2.1. Dataset Description and Statistics

ENA24 images are captured in the wild under real monitoring conditions: variable lighting (including night shots), motion blur, partial animals, occlusion, and cluttered backgrounds. Annotations provide axis-aligned boxes and **species-level category ids** in the raw COCO file; how we use those labels in training is defined in Section 3.

**Full subset used in this project (`data/ena24_full`):**

| Item | Value |
| --- | --- |
| Images | 8,789 |
| Bounding-box annotations (all splits) | 9,772 |
| Species categories in annotations | 22 |
| Images with multiple boxes | common (annotation count > image count) |

Annotations are not always one box per image: many frames contain several animals, which matches the variable $M_i$ formulation in Section 1.

**Why the data is difficult for detection:** burst sequences (many near-identical consecutive frames), small or distant animals, and domain shift between locations and seasons. Burst structure also affects how we split train/val/test — see Section 5, not repeated here.


### 2.2. Development Sample vs Full Dataset

We use two on-disk copies of ENA24 with the same COCO schema:

| | `data/ena24_sample` | `data/ena24_full` |
| --- | --- | --- |
| **Purpose** | Fast iteration, smoke tests, notebook exploration | Final training and reported metrics |
| **Size** | 31 images in our copy (script default: 30) | 8,789 images |
| **Created by** | `scripts/prepare_ena24_sample.py` (random subset, seed 42; downloads from LILA or copies from a local ENA24 folder) | Full download / local ENA24 layout |
| **Train/val/test** | Random per-image split in configs (`split_strategy` not set → random) | Group-level split via `split_manifest.json` (Section 5) |
| **Configs** | `baseline_config.json`, `yolo_config.json` | `baseline_config_full.json`, `yolo_config_full.json` |

The sample is **not** a statistically representative subset of the full dataset; it only validates that the pipeline runs. Config `data.size` may further cap how many sample images a run uses (e.g. baseline sample: 20, YOLO sample: 60). Reported comparison numbers for the full project target the **full dataset with the no-leak split** (Section 10).


## 3. Project Assumptions and Scope

This section fixes what we **commit to build and evaluate** in the semester project. Problem notation (Section 1) and dataset facts (Section 2) are broader; here we narrow the task to a single detection setting and two model families on the same splits.


### 3.1. Single-Class Detection (Bounding Boxes Only)

**Goal:** predict **where** animals are, not **which species** they are.

ENA24 COCO files contain many species `category_id` values (Section 2). We deliberately collapse all animals into one class:

| Component | How single-class is applied |
| --- | --- |
| **Baseline** | Binary window classifier: positive crop = overlaps a GT box; negative = background. No species head. |
| **YOLO** | `prepare_yolo_dataset.py` writes every box as class `0` (`nc: 1`, name `object`). `--multi_class` exists but is not used in our experiments. |
| **Metrics** | `src/detection/detection_metrics.py` matches predictions to GT by **IoU only**; predicted class id is ignored. |

Success is measured by **localization**: precision, recall, F1, mAP at IoU 0.5 (Section 8). A prediction counts as correct if it overlaps a ground-truth box sufficiently, regardless of species label in the annotation file.


### 3.2. Baseline vs Target Model (YOLO)

We compare two detectors on the **same train/val/test splits** (Section 5) and **same metrics** (Section 8):

| | **Baseline** | **Target model (YOLOv8n)** |
| --- | --- | --- |
| **Role** | Classical reference pipeline from the course | Modern single-stage detector |
| **Idea** | Train object vs background on crops → scan image with sliding windows → NMS | End-to-end bounding-box regression + classification head (one class) |
| **Implementation** | ResNet18 + sliding window + NMS (Section 6) | Ultralytics YOLOv8n (Section 7) |
| **Why both** | Shows a minimal NN-based detector and highlights the cost of window search | Expected stronger accuracy and much faster inference on full images |

The baseline is not meant to match state-of-the-art detection; it anchors the project methodologically. YOLO is the primary model for final quantitative results on the full dataset.


### 3.3. Out of Scope

The following are **not** part of the current project deliverables:

- **Species or multi-class detection** — per-species boxes and metrics unless explicitly enabled via `--multi_class` (not used in our runs).
- **Classification without detection** — image-level species labels only.
- **Human images** — we use the non-human ENA24 subset only.
- **Video / temporal modeling** — each image is treated independently; burst handling is for **split integrity**, not sequence prediction.
- **Production deployment** — no API, edge pipeline, or real-time camera integration.
- **Heavy hyperparameter search** — fixed configs in `src/config/`; limited tuning documented for pHash grouping only (Section 5).

Species labels in ENA24 remain available for **future work** (detect then classify); Section 13 lists possible extensions.


## 4. Solution Architecture

The repository implements one **shared data and evaluation core** with two **training branches** (baseline and YOLO). Both branches read the same COCO annotations and the same train/val/test assignment on the full dataset; both report metrics through the same detection module.


### 4.1. Pipeline Overview (Data to Metrics)

```mermaid
flowchart LR
  subgraph data [Data]
    ENA24[ENA24 COCO]
    Sample[ena24_sample]
    Full[ena24_full]
    Meta[metadata: groups + split_manifest]
  end

  subgraph anti [Anti-leak full only]
    pHash[pHash grouping]
    Split[group-level split]
  end

  subgraph baseline [Baseline branch]
    BTrain[train_model.py]
    BEval[evaluate_baseline.py]
    BCKPT[baseline_resnet*_best.pt]
  end

  subgraph yolo [YOLO branch]
    YPrep[prepare_yolo_dataset.py]
    YTrain[train_yolo.py]
    YEval[evaluate_yolo.py]
    YCKPT[yolo*_best.pt]
  end

  subgraph shared [Shared]
    Metrics[detection_metrics.py]
  end

  ENA24 --> Sample
  ENA24 --> Full
  Full --> pHash --> Split --> Meta
  Meta --> BTrain
  Meta --> YPrep
  Sample --> BTrain
  Sample --> YPrep
  BTrain --> BCKPT --> BEval
  YPrep --> YTrain --> YCKPT --> YEval
  BEval --> Metrics
  YEval --> Metrics
```

**Typical workflow (full dataset):**

1. **Prepare data** — full ENA24 on disk (`data/ena24_full/images/`, annotations JSON).
2. **Anti-leak split** — `build_duplicate_groups.py` → `split_coco_by_groups.py` → `split_manifest.json` (Section 5).
3. **Baseline** — `train_model.py` (Lightning) exports CNN weights → `evaluate_baseline.py` runs sliding window + NMS on val/test images.
4. **YOLO** — `prepare_yolo_dataset.py` builds Ultralytics layout → `train_yolo.py` → `evaluate_yolo.py` on the same splits.
5. **Compare** — both eval scripts call `src/detection/detection_metrics.py` (IoU 0.5 matching, class-agnostic).

Sample dataset skips step 2 and uses a random per-image split from config for smoke tests only (Section 2.2).


### 4.2. Repository Structure and Configuration Files

**Top-level layout (code):**

| Path | Role |
| --- | --- |
| `scripts/` | CLI entry points: data prep, anti-leak split, train, evaluate |
| `src/datasets/` | COCO I/O, splits, pHash grouping, window crops, Lightning DataModule |
| `src/models/baseline/` | ResNet/CNN classifiers, sliding window, Lightning module |
| `src/detection/` | IoU, NMS, shared metrics, bbox visualization |
| `src/config/` | JSON experiment configs + `load_config.py`, `paths.py` |
| `src/utils/` | W&B helpers (YOLO / legacy baseline script) |

**Local artifacts (not in git; paths from configs):**

| Path | Contents |
| --- | --- |
| `data/ena24_sample`, `data/ena24_full` | COCO images + annotations |
| `data/ena24_full/metadata/` | `group_manifest.json`, `split_manifest.json`, split statistics |
| `data/ena24_yolo`, `data/ena24_yolo_full` | Ultralytics `images/`, `labels/`, `data.yaml` |
| `checkpoints/` | Best `.pt` weights for baseline and YOLO |
| `logs/` | Lightning runs for baseline |
| `runs/detect/` | Ultralytics training logs for YOLO |
| `results/` | Comparison tables, eval JSON (e.g. `yolo_full_test.json`) |

**Configuration pairs (sample vs full):**

| Config | Dataset | Split strategy |
| --- | --- | --- |
| `baseline_config.json` | `ena24_sample` | random (`data.size`: 20) |
| `baseline_config_full.json` | `ena24_full` | `manifest` → `split_manifest.json` |
| `yolo_config.json` | `ena24_sample` | random (`data.size`: 60) |
| `yolo_config_full.json` | `ena24_full` | `manifest` |

Paths in JSON use `../data/...` relative to `scripts/`; `normalize_config_paths()` resolves them to the project root. Training scripts accept `--config_path`; seeds and ratios live in the JSON files for reproducibility.


## 5. Data Leakage Problem and Fix

On the full ENA24 set, a **naive random train/val/test split per image** is not a valid evaluation protocol: many frames from the same burst event look almost identical. Without fixing the split, models can score well on test images that are near-duplicates of training frames. This section describes the problem and the **group-level split** we use for all full-dataset experiments (baseline and YOLO).


### 5.1. Source of Leakage: Burst Sequences in Camera Trap Data

Camera traps often record **burst sequences**: when motion is detected, the device saves many consecutive frames of the same animal in the same pose and background. In ENA24 these bursts are visually very similar but are stored as separate images with independent file names.

The public COCO metadata for this release does **not** provide a reliable `sequence_id` or timestamp field for grouping bursts. A random 60/20/20 split therefore assigns frames from one event to **both** train and test. The model is evaluated on images it has effectively already seen, which **inflates** precision, recall, and mAP.

This is **data leakage** at the split level—not label noise in individual boxes. Section 10.4 shows how much YOLO metrics drop when we switch from a leaky split to the group-level split.


### 5.2. Naive Split vs Group-Level Split

| | **Naive split** | **Group-level split (ours)** |
| --- | --- | --- |
| **Unit of splitting** | Individual images | Groups of visually similar images |
| **Burst handling** | Frames from one event can land in train and test | All images in a group share one split label |
| **Leakage risk** | High on camera trap data | Low if groups approximate bursts |
| **Used for** | `ena24_sample` smoke tests only | `ena24_full` training and reported metrics |
| **Config flag** | default random split in sample configs | `split_strategy: "manifest"` in `*_full.json` |

The naive split is acceptable on ~20–30 sample images where burst overlap is unlikely. It is **not** used for final numbers on the full dataset.


### 5.3. Method: pHash, Grouping, and Train / Val / Test Split

**Step A — Perceptual hashing (pHash)**  
Each image is hashed with [`imagehash`](https://pypi.org/project/ImageHash/) after cropping **6%** from the top and **10%** from the bottom to remove timestamp/camera overlays that would otherwise dominate the hash. Encodings are cached in `phash_encodings.json` for reproducibility.

**Step B — Pairwise matching + Union-Find**  
Pairs with Hamming distance ≤ **3** (chosen after a production sweep; see `results/phash_production_sweep.json`) are linked. **Union-Find** merges pairs into groups, with **max group size 50** to cap oversized clusters (typical burst length in ENA24).

**Step C — Group-level split**  
Whole groups are assigned to train / val / test with ratios **60 / 20 / 20** and **seed 42**. Output: `group_manifest.json` (image → group id) and `split_manifest.json` (image → split name).

**Step D — Consumption**  
Baseline, YOLO prepare, and both eval scripts read `split_manifest.json` when `split_strategy` is `manifest`—one source of truth for train / val / test (Section 4).


### 5.4. Split Validation and Statistics

After building the manifest we verify that **no group spans multiple splits** (`leakage_check_passed: true` in `split_statistics.json`).

**Full dataset (`ena24_full`) summary:**

| Metric | Value |
| --- | --- |
| Images | 8,789 |
| Groups | 5,143 |
| Singleton groups (1 image) | 3,981 |
| Largest group | 50 images |
| Images in multi-image groups | 54.7% |
| Train / val / test images | 5,417 / 1,640 / 1,732 |
| Train / val / test groups | 3,085 / 1,028 / 1,030 |

`report_split_statistics.py --compare_random` simulates naive random splits and reports **mean leaking groups**—how many burst groups would be broken across splits under the old protocol. That number should be high relative to a correct group split, confirming the naive approach would leak.

Optional visual check: `preview_split_samples.py` copies example images to `metadata/split_previews/` for manual inspection.


### 5.5. Commands — Anti-Leakage Pipeline

Run from **project root** with venv active. Prerequisites: `data/ena24_full/images/` and a COCO JSON annotations file.

**1. Group near-duplicates** (~15–45 min first run; cached on reruns):

```bash
python scripts/build_duplicate_groups.py --data_dir data/ena24_full --max_distance_threshold 3
```

**2. Split groups into train / val / test** (expect `Leakage check: True` in output):

```bash
python scripts/split_coco_by_groups.py --data_dir data/ena24_full --group_manifest data/ena24_full/metadata/group_manifest.json --seed 42 --train_ratio 0.6 --val_ratio 0.2 --export_coco_splits
```

**3. Verify statistics and naive-split comparison:**

```bash
python scripts/report_split_statistics.py --data_dir data/ena24_full --group_manifest data/ena24_full/metadata/group_manifest.json --split_manifest data/ena24_full/metadata/split_manifest.json --compare_random
```

**4. Optional — visual spot-check:**

```bash
python scripts/preview_split_samples.py --data_dir data/ena24_full --copy --per_group
```

**5. Optional — parameter sweep** (only if grouping looks wrong):

```bash
python scripts/sweep_phash_production_params.py --data_dir data/ena24_full --thresholds 3 4 5 6 --max_group_sizes 40 50 60 --export_previews
```

**Key output files** under `data/ena24_full/metadata/`:

`group_manifest.json`, `split_manifest.json`, `split_statistics.json`, `phash_encodings.json`, `hash_config.json`, `duplicate_pairs.json`, optional `splits/` and `split_previews/`.


## 6. Baseline: ResNet + Sliding Window + NMS

The baseline follows a **classical detection pipeline**: train a binary classifier on fixed-size crops, then scan each full image with sliding windows, keep high-confidence regions, and merge overlaps with NMS. It implements the single-class scope (Section 3) without predicting species. Training and evaluation are wired through PyTorch Lightning (`scripts/train_model.py`, `LitSlidingWindowCNNDetector`).


### 6.1. Method Description

```mermaid
flowchart LR
  Train[COCO images + boxes]
  Crops[Positive / negative crops 128×128]
  CNN[ResNet18 binary head]
  SW[Sliding window on full image]
  NMS[NMS]
  Det[Predicted boxes]

  Train --> Crops --> CNN
  CNN --> SW --> NMS --> Det
```

**Stages:**

1. **Train** a window classifier: object vs background on 128×128 crops (Section 6.2).
2. **Detect** on each val/test image: place windows at multiple scales, score each window with the CNN (Section 6.3).
3. **Post-process** overlapping high-score windows with NMS.
4. **Evaluate** with the shared detection metrics module (Section 8).

Unlike YOLO, the baseline does **not** regress box coordinates directly; window positions define candidate boxes. This is slow on large images but easy to interpret and aligns with course-style “classifier + search” baselines.


### 6.2. Window Classifier Training (Positive / Negative Crops)

**Model:** `ResNet18` with ImageNet weights; all backbone layers frozen; final `fc` replaced by a single logit (binary). Optimizer: Adam, `BCEWithLogitsLoss`. Alternative `simple_cnn` exists in code but all reported runs use `model: "resnet"`.

**Training data** (`ENA24WindowDataset`):

| Crop type | How it is built |
| --- | --- |
| **Positive** | One 128×128 crop per ground-truth box (crop aligned to annotated animal region). Label = 1. |
| **Negative** | Random windows with max IoU to any GT box &lt; **0.2**. Count = **3×** number of positives per image. Label = 0. |

Crops are resized to 128×128 with ImageNet normalization (same as inference).

**Lightning training** (`scripts/train_model.py`):

- **Crop-level val** (dataloader 0): loss and accuracy on crops; checkpoint monitors `cnn_val_loss`.
- **Full-image val** (dataloader 1): sliding window + NMS detection metrics logged during training (slow).

After training, the best Lightning checkpoint is exported to a plain CNN `state_dict` at `cnn_training.best_checkpoint_path` for `evaluate_baseline.py`.


### 6.3. Detection: Sliding Window, Thresholds, and NMS

**Sliding window** (`SlidingWindow` in `sliding_window_detection.py`):

| Parameter | Value (configs) | Effect |
| --- | --- | --- |
| `window_sizes` | `[192, 256]` | Square windows in pixels on the original image |
| `overlap_ratio` | `0.7` | Stride = window_size × (1 − overlap); dense grid of candidates |
| `threshold` | `0.7` | Min sigmoid score to keep a window as detection |
| `crop_size` | `128` | Resize each window before CNN forward pass |

For each kept window, the **window rectangle** is the predicted box (not a refined inner box).

**NMS** (`src/detection/nms.py`): suppress overlapping predictions with IoU &gt; **0.3** between candidates (config `nms.iou_threshold`).

**Separate thresholds:**

| Setting | Value | Used for |
| --- | --- | --- |
| `cnn_training.threshold` | `0.8` | Crop-level train/val accuracy during Lightning |
| `sliding_window.threshold` | `0.7` | Which windows become detections |
| `metrics.iou_threshold` | `0.5` | TP/FP/FN matching vs ground truth (Section 8) |

**Inference cost:** every window at two scales is a forward pass—evaluation on the full test split (~1,732 images) is much slower than YOLO, especially on CPU.


### 6.4. Configuration (`baseline_config.json` / `baseline_config_full.json`)

| Key | Sample config | Full config |
| --- | --- | --- |
| `data.data_dir` | `ena24_sample` | `ena24_full` |
| `data.size` | `20` | all images (no cap) |
| Split | random 60/20/20 | `split_strategy: manifest` + `split_manifest.json` |
| `cnn_training.num_epochs` | 4 | 10 |
| `cnn_training.batch_size` | 128 | 128 |
| `cnn_training.learning_rate` | 0.001 | 0.001 |
| `cnn_training.best_checkpoint_path` | `baseline_resnet_best.pt` | `baseline_resnet_full_best.pt` |
| Sliding window / NMS / metrics | same | same |

**Commands:**

```bash
# Sample
python scripts/train_model.py --config_path src/config/baseline_config.json
python scripts/evaluate_baseline.py --config_path src/config/baseline_config.json --split test

# Full (requires split_manifest from Section 5)
python scripts/train_model.py --config_path src/config/baseline_config_full.json
python scripts/evaluate_baseline.py --config_path src/config/baseline_config_full.json --split test
```

Legacy all-in-one script: `scripts/run_baseline_model.py` (sample config only, optional W&B). Primary path: `train_model.py` + `evaluate_baseline.py`.


## 7. Target Model: YOLOv8n (Ultralytics)

The target detector is **YOLOv8n** via the [Ultralytics](https://github.com/ultralytics/ultralytics) library: a single-stage network that predicts boxes and class scores in one forward pass. We use the nano variant (`yolov8n.pt`) for faster training on the full dataset. Training and inference are wrapped in project scripts; the model architecture itself is not reimplemented in `src/models/`.


### 7.1. Method Description and YOLO Data Preparation

**Detection flow:**

1. **Prepare** — `scripts/prepare_yolo_dataset.py` converts ENA24 COCO to Ultralytics layout.
2. **Train** — `scripts/train_yolo.py` calls `YOLO.train()` on `data.yaml`, copies `best.pt` to `checkpoints/`.
3. **Evaluate** — `scripts/evaluate_yolo.py` runs inference on val/test and scores with `detection_metrics.py` (same as baseline).

**Data layout** (under `data/ena24_yolo` or `data/ena24_yolo_full`):

```text
images/{train,val,test}/   # symlinks or copies of COCO images
labels/{train,val,test}/   # one .txt per image (YOLO normalized format)
data.yaml                  # paths, nc, class names
split_summary.json         # split counts for reproducibility
```

**Single-class mode (default):** every COCO annotation is written as class **0** (`nc: 1`, name `object`). Species `category_id` values are ignored—aligned with the baseline (Section 3). Multi-class export is available via `--multi_class` but not used in our experiments.

**Splits:** the prepare script uses the same `prepare_data_splits` logic as the baseline (`seed`, ratios, optional `split_manifest.json` on full). Baseline and YOLO therefore see the **same images** in train / val / test when configs match.

**Inference post-processing:** Ultralytics applies internal NMS; our eval uses `conf_threshold` and `iou_threshold` from config (Section 7.2). Final TP/FP/FN still use **match IoU 0.5** from `metrics.iou_threshold` (Section 8).

**Experiment tracking:** W&B logging is enabled in both YOLO configs (`wandb.project: ena24-yolo-detector-v2`).


### 7.2. Configuration (`yolo_config.json` / `yolo_config_full.json`)

| Key | Sample (`yolo_config.json`) | Full (`yolo_config_full.json`) |
| --- | --- | --- |
| `data.coco_dir` | `ena24_sample` | `ena24_full` |
| `data.output_dir` | `ena24_yolo` | `ena24_yolo_full` |
| `data.size` | 60 | all images (`null`) |
| Split | random 60/20/20 | `manifest` + `split_manifest.json` |
| `model.weights` | `yolov8n.pt` | `yolov8n.pt` |
| `training.epochs` | 10 | 20 |
| `training.imgsz` | 640 | 640 |
| `training.batch` | 16 | 16 |
| `training.name` | `ena24_yolo_sample` | `ena24_yolo_full` |
| `inference.conf_threshold` | 0.02 | 0.25 |
| `inference.iou_threshold` | 0.45 (NMS) | 0.45 |
| `metrics.iou_threshold` | 0.5 | 0.5 |
| Checkpoint | `yolo_best.pt` | `yolo_full_best.pt` |
| Ultralytics logs | `runs/detect/ena24_yolo_sample/` | `runs/detect/ena24_yolo_full/` |

Lower `conf_threshold` on the sample helps surface predictions on tiny data; full eval uses **0.25** (standard default).

**Commands:**

```bash
# Sample
python scripts/prepare_yolo_dataset.py --config_path src/config/yolo_config.json
python scripts/train_yolo.py --config_path src/config/yolo_config.json
python scripts/evaluate_yolo.py --config_path src/config/yolo_config.json --split test

# Full (requires split_manifest from Section 5)
python scripts/prepare_yolo_dataset.py --config_path src/config/yolo_config_full.json
python scripts/train_yolo.py --config_path src/config/yolo_config_full.json
python scripts/evaluate_yolo.py --config_path src/config/yolo_config_full.json --split test \
  --output_json results/yolo_full_test.json
```

On Windows, if image symlinks fail during prepare, add `--copy_images`. Optional: `train_yolo.py --prepare` combines prepare + train; `--eval_after_train` runs eval after training.


## 8. Evaluation Metrics

Baseline and YOLO are scored with the **same module** (`src/detection/detection_metrics.py`) so tables in Section 10 are directly comparable. Matching is **class-agnostic**: a prediction is correct if it overlaps a ground-truth box sufficiently, regardless of species labels in the COCO file (Section 3).


### 8.1. Shared Detection Metrics (Class-Agnostic)

**Per-image matching** (`count_detection_results`):

1. Sort predictions by **confidence** (descending).
2. For each prediction, assign it to the best **unmatched** ground-truth box if IoU ≥ match threshold.
3. Count **TP** (matched), **FP** (no match), **FN** (unmatched GT boxes).

Each ground-truth box can match **at most one** prediction (greedy, score-sorted)—standard for detection eval at a single IoU threshold.

**Aggregated metrics** (summed over all images in a split):

| Metric | Definition |
| --- | --- |
| **Precision** | TP / (TP + FP) |
| **Recall** | TP / (TP + FN) |
| **F1** | 2 · precision · recall / (precision + recall) |
| **mAP @ IoU τ** | Mean average precision at match threshold τ: global sort of all predictions by score, build precision–recall curve, integrate with interpolated precision (VOC-style AP at one IoU) |
| **Mean IoU** | For each GT box, IoU of its best overlapping prediction (even if below τ); averaged over GT boxes — localization quality separate from TP/FP threshold |

**Box format:** `[x1, y1, x2, y2]` in pixel coordinates (top-left and bottom-right). IoU uses axis-aligned intersection over union (`src/detection/IoU.py`).

**Scripts:** `evaluate_baseline.py` and `evaluate_yolo.py` both call `evaluate_detections`, `calculate_map`, and `mean_iou` with the same `metrics.iou_threshold` from config.


### 8.2. IoU, Confidence, and NMS Thresholds

Three different thresholds appear in the pipeline—**do not confuse them**:

| Threshold | Config key | Typical value | Role |
| --- | --- | --- | --- |
| **Match IoU** | `metrics.iou_threshold` | **0.5** | Defines TP vs FP when comparing predictions to GT (both models, all reported P/R/F1/mAP) |
| **NMS IoU** | `nms.iou_threshold` (baseline) or `inference.iou_threshold` (YOLO) | **0.3** / **0.45** | Suppresses overlapping *predictions* before scoring—not used for TP/FP |
| **Detection confidence** | `sliding_window.threshold` (baseline) or `inference.conf_threshold` (YOLO) | **0.7** / **0.25** (full) | Minimum score to emit a prediction |

```mermaid
flowchart LR
  Model[Model output]
  Conf[Confidence filter]
  NMS[NMS]
  Match[Match IoU 0.5]
  Metrics[P / R / F1 / mAP]

  Model --> Conf --> NMS --> Match --> Metrics
```

**Baseline-specific:** `cnn_training.threshold` (**0.8**) is only for crop-level accuracy during Lightning training—not for sliding-window detection scores.

**Alignment across models:** both eval configs set `metrics.iou_threshold: 0.5`. YOLO full eval uses `conf_threshold: 0.25` and NMS IoU `0.45`; baseline uses sliding-window threshold `0.7` and NMS IoU `0.3` (Section 6.3). Different post-processing is expected; the **match** step is what makes metrics comparable.


## 9. Experiments and Commands

This section lists **what we ran** and **how to reproduce it**. Method details are in Sections 5–8; **numeric results** are in Section 10. All commands assume the **project root** as working directory.


### 9.1. Environment and Data Setup

```bash
python -m venv .venv
# Windows: .venv\Scripts\activate
# Linux/macOS: source .venv/bin/activate

pip install -r requirements.txt
```

**Key dependencies:** PyTorch, Lightning, Ultralytics, OpenCV, `imagehash` (anti-leak pipeline), W&B.

**Data layout:**

| Path | Contents |
| --- | --- |
| `data/ena24_sample/` | Small dev sample (`scripts/prepare_ena24_sample.py`) |
| `data/ena24_full/images/` + annotations JSON | Full ENA24 for production runs |
| `data/ena24_full/metadata/` | Group/split manifests after Section 5 pipeline |

**Create dev sample** (public download or local `--data_dir`):

```bash
python scripts/prepare_ena24_sample.py
# or: python scripts/prepare_ena24_sample.py --data_dir PATH_TO_FULL_ENA24
```

**Full dataset anti-leak split** (required before full experiments — commands in Section 5.5):

```bash
python scripts/build_duplicate_groups.py --data_dir data/ena24_full --max_distance_threshold 3
python scripts/split_coco_by_groups.py --data_dir data/ena24_full \
  --group_manifest data/ena24_full/metadata/group_manifest.json --seed 42 \
  --train_ratio 0.6 --val_ratio 0.2 --export_coco_splits
```


### 9.2. Experiment: Development Sample (`ena24_sample`)

**Purpose:** smoke test — verify training, eval, and metrics on ~20–60 images. **Random per-image split** (not used for final reported numbers).

| Setting | Value |
| --- | --- |
| Config data cap | baseline `size: 20`, YOLO `size: 60` |
| Split | seed 42, 60% / 20% / 20% |
| Eval split | test (typically 4 images when `size: 20`) |


#### 9.2.1. Baseline — Training and Evaluation (Sample)

```bash
python scripts/train_model.py --config_path src/config/baseline_config.json
python scripts/evaluate_baseline.py --config_path src/config/baseline_config.json --split test
```

| Item | Value |
| --- | --- |
| Config | `src/config/baseline_config.json` |
| Epochs | 4 |
| Checkpoint | `checkpoints/baseline_resnet_best.pt` |
| Results | Section 10.1 |


#### 9.2.2. YOLO — Prepare, Train, and Evaluate (Sample)

```bash
python scripts/prepare_yolo_dataset.py --config_path src/config/yolo_config.json
python scripts/train_yolo.py --config_path src/config/yolo_config.json
python scripts/evaluate_yolo.py --config_path src/config/yolo_config.json --split test
```

| Item | Value |
| --- | --- |
| Config | `src/config/yolo_config.json` |
| Epochs | 10 |
| `conf_threshold` | 0.02 |
| Checkpoint | `checkpoints/yolo_best.pt` |
| Ultralytics run | `runs/detect/ena24_yolo_sample/` |
| Results | Section 10.1 |


### 9.3. Experiment: Full ENA24 (`ena24_full`, No-Leak Split)

**Purpose:** main experiments for the final report. Both models use `split_strategy: manifest` and `data/ena24_full/metadata/split_manifest.json` (Section 5).

| Setting | Value |
| --- | --- |
| Images | 8,789 total → train 5,417 / val 1,640 / test 1,732 |
| Match IoU (metrics) | 0.5 |
| Primary eval split | **test** (no leak) |


#### 9.3.1. Baseline — Training and Evaluation (Full)

```bash
python scripts/train_model.py --config_path src/config/baseline_config_full.json
python scripts/evaluate_baseline.py --config_path src/config/baseline_config_full.json --split test
```

| Item | Value |
| --- | --- |
| Config | `src/config/baseline_config_full.json` |
| Epochs | 10 |
| Checkpoint | `checkpoints/baseline_resnet_full_best.pt` |
| Lightning logs | `logs/baseline_sliding_window_cnn_*` |
| Results | Section 10.2.1 (*pending if training not finished*) |

**Note:** sliding-window eval on ~1,732 test images is slow; GPU recommended for training and evaluation.


#### 9.3.2. YOLO — Prepare, Train, and Evaluate (Full)

```bash
python scripts/prepare_yolo_dataset.py --config_path src/config/yolo_config_full.json
python scripts/train_yolo.py --config_path src/config/yolo_config_full.json
python scripts/evaluate_yolo.py --config_path src/config/yolo_config_full.json --split test \
  --output_json results/yolo_full_test.json
```

On Windows, add `--copy_images` to `prepare_yolo_dataset.py` if symlinks fail.

| Item | Value |
| --- | --- |
| Config | `src/config/yolo_config_full.json` |
| Epochs | 20 |
| `conf_threshold` | 0.25 |
| Checkpoint | `checkpoints/yolo_full_best.pt` |
| Ultralytics run | `runs/detect/ena24_yolo_full/` |
| Eval JSON | `results/yolo_full_test.json` |
| Results | Section 10.2.2 |


### 9.4. Comparative Experiment: YOLO With vs Without Data Leakage

**Purpose:** quantify how a **naive random split** inflates YOLO metrics compared to the **group-level split** (Section 5).

| Run | Split | Eval split | Config / notes |
| --- | --- | --- | --- |
| **Leaky (early run)** | Random per image, same 60/20/20 ratios | **val** (1,757 images) | Before `split_manifest`; trained with default random split |
| **No leak (reported)** | `split_manifest.json` | **test** (1,732 images) | `yolo_config_full.json` as in 9.3.2 |

These two rows are **not** a like-for-like split comparison (val ≠ test). They illustrate the *direction* of leakage bias: higher P/R/F1/mAP when near-duplicates can appear in train and eval.

**No-leak commands:** Section 9.3.2.

**Leaky run:** train/eval with full YOLO config **before** applying the anti-leak pipeline, or temporarily remove `split_strategy: manifest` and use random split — same `train_yolo.py` / `evaluate_yolo.py` with `--split val`.

**Results:** Section 10.4.


## 10. Results

Numbers below come from `evaluate_baseline.py` and `evaluate_yolo.py` with **match IoU 0.5** (Section 8). Sample runs validate the pipeline; **full no-leak test** is the primary quantitative result for YOLO. Full baseline on the same test split is **pending** (Section 10.2.1).


### 10.1. Results Table — Development Sample (Smoke Test)

**Setup:** `ena24_sample`, baseline `size: 20` → **4 test images**; seed 42; CPU; checkpoints `baseline_resnet_best.pt`, `yolo_best.pt`.

| Metric | Baseline | YOLOv8n |
| --- | --- | --- |
| Images | 4 | 4 |
| TP | 0 | 0 |
| FP | 2 | 0 |
| FN | 4 | 4 |
| Precision | 0.00 | 0.00 |
| Recall | 0.00 | 0.00 |
| F1 | 0.00 | 0.00 |
| mAP @ IoU 0.5 | 0.00 | 0.00 |
| Mean IoU | 0.00 | 0.00 |
| Avg detections / image | 0.50 | 0.00 |

**Per-image (test):**

| Image | GT | Baseline det. | Baseline TP/FP/FN | YOLO det. | YOLO TP/FP/FN |
| --- | --- | --- | --- | --- | --- |
| 2505.jpg | 1 | 2 | 0 / 2 / 1 | 0 | 0 / 0 / 1 |
| 5140.jpg | 1 | 0 | 0 / 0 / 1 | 0 | 0 / 0 / 1 |
| 9249.jpg | 1 | 0 | 0 / 0 / 1 | 0 | 0 / 0 / 1 |
| 5807.jpg | 1 | 0 | 0 / 0 / 1 | 0 | 0 / 0 / 1 |

**Takeaway:** neither model scored a TP on N=4 — expected for a smoke test, not for model ranking. Confirms end-to-end eval runs; see Section 9.2 for commands.


### 10.2. Results Table — Full Dataset, Test Split (No Leakage)

**Shared setup:** `ena24_full`, group-level split (`split_manifest.json`), evaluated on **test** (1,732 images, 1,935 GT boxes), match IoU 0.5.


#### 10.2.1. Baseline (Full)

*Not yet evaluated — fill after `evaluate_baseline.py` with `baseline_config_full.json` on `--split test`.*

| Metric | Baseline (ResNet + SW) |
| --- | --- |
| Config | `baseline_config_full.json` |
| Test images | 1,732 |
| TP / FP / FN | — |
| Precision / Recall / F1 | — |
| mAP @ IoU 0.5 | — |
| Mean IoU | — |


#### 10.2.2. YOLO (Full)

**Run:** `yolo_config_full.json`, 20 epochs, `yolov8n.pt`, imgsz 640, batch 16; `conf_threshold` 0.25, NMS IoU 0.45; checkpoint `yolo_full_best.pt`; eval 2026-06-12 (`results/yolo_full_test.json`).

| Metric | YOLOv8n |
| --- | --- |
| Images | 1,732 |
| TP | 1,629 |
| FP | 77 |
| FN | 306 |
| Precision | 0.955 |
| Recall | 0.842 |
| F1 | 0.895 |
| mAP @ IoU 0.5 | 0.837 |
| Mean IoU | 0.738 |
| Avg detections / image | 0.98 |


### 10.3. Baseline vs YOLO Comparison (Same Test Split)

Side-by-side on **no-leak test** (1,732 images). Baseline column pending Section 10.2.1.

| Metric | Baseline (full) | YOLOv8n (full) |
| --- | --- | --- |
| Precision | — | 0.955 |
| Recall | — | 0.842 |
| F1 | — | 0.895 |
| mAP @ IoU 0.5 | — | 0.837 |
| Mean IoU | — | 0.738 |

*Update this table once baseline full eval completes.*


### 10.4. Impact of Data Leakage on YOLO Metrics

Same YOLOv8n architecture; different **split protocol**. Not a strict A/B on the same eval split (leaky run used **val**, no-leak run used **test**) — shows direction of bias from burst leakage.

| Setting | Leaky (random split) | No leak (group split) |
| --- | --- | --- |
| Eval split | val (1,757 img) | test (1,732 img) |
| Evaluated | 2026-06-02 | 2026-06-12 |
| Precision | 0.980 | 0.955 |
| Recall | 0.939 | 0.842 |
| F1 | 0.959 | 0.895 |
| mAP @ IoU 0.5 | 0.938 | 0.837 |
| Mean IoU | 0.819 | 0.738 |
| TP / FP / FN | 1833 / 38 / 119 | 1629 / 77 / 306 |

Leaky metrics are **systematically higher** (e.g. F1 0.96 vs 0.89, mAP 0.94 vs 0.84), consistent with near-duplicate frames in both train and eval. The no-leak test scores are the ones we treat as the honest generalization estimate (Section 11.2).


### 10.5. Visualizations (Example Predictions)

*To add:* side-by-side GT vs baseline vs YOLO on a few test images (successes and failures). Scripts exist locally (`scripts/visualize_baseline_pred.py`, `scripts/visualize_yolo.py`, gitignored) or export from `src/detection/bbox_visualization.py`.

Suggested cases after baseline eval: high-confidence TP, missed small animal (FN), duplicate-window FP on baseline.


## 11. Results Interpretation

This section interprets **YOLOv8n on full ENA24** (Section 10.2.2) and what the **anti-leak split** changes in reported metrics (Section 10.4). Baseline comparison is deferred until full baseline eval is available (Section 10.2.1).


### 11.1. YOLO Performance on the No-Leak Test Set

On **1,732 held-out test images** (whole burst groups unseen during training), YOLOv8n reaches **precision 0.95**, **recall 0.84**, **F1 0.89**, and **mAP 0.84** at IoU 0.5. These are strong numbers for wild camera-trap imagery with variable lighting, scale, and occlusion.

**What the metrics suggest:**

- **High precision (77 FP on ~1,700 images)** — the model rarely fires on empty background; false alarms are not the main failure mode at `conf_threshold` 0.25.
- **Recall below precision (306 FN)** — missed animals are the larger error bucket: ~16% of ground-truth boxes have no matching prediction above the match threshold.
- **Mean IoU 0.74** — when a GT box does get a overlapping prediction, localization is generally reasonable, but not as high as mAP alone might suggest; some matches are barely above 0.5 IoU.
- **~0.98 detections per image** — close to one box per image on average, which fits a dataset where many frames contain a single animal, though multi-animal images exist (Section 2).

**Training curve (Ultralytics val):** mAP50 on the validation split rose from ~0.32 (epoch 1) to ~0.49 (epoch 20) on the **no-leak val set** (`runs/detect/ena24_yolo_full/results.csv`). End-to-end test metrics (Section 10.2.2) use our shared evaluator, not Ultralytics’ internal metric definition, but the trend confirms the model learned useful features over 20 epochs.

Overall, YOLO is a **usable single-class detector** on this data under an honest split; remaining error is dominated by **missed detections**, not clutter false positives.


### 11.2. Importance of Fixing Data Leakage

Section 10.4 compares two YOLO training/eval regimes on the same architecture:

| | Leaky (random split, val) | No leak (group split, test) |
| --- | --- | --- |
| F1 | 0.959 | 0.895 |
| mAP @ 0.5 | 0.938 | 0.837 |
| FN | 119 | **306** |

**Why the gap is real:** camera trap bursts produce dozens of near-duplicate frames per event. A random image split lets the model **train and evaluate on siblings** from the same burst. Metrics then measure memorization of visual repetition, not generalization to new events.

**What the group split fixes:** pHash grouping + splitting whole groups ensures test images are not near-duplicates of training frames (Section 5). `leakage_check_passed: true` confirms no group spans splits.

**How to read the drop:** lower recall and mAP on the no-leak test are **expected** and **more trustworthy** for reporting. The leaky run’s F1 ~0.96 would overstate deployment performance if bursts were not handled.

**For this project:** all **final** YOLO numbers in Section 10.2.2 use the no-leak test split. The leaky run is kept as a **methodological contrast**—it demonstrates why burst-aware splitting is part of correct experimental design on ENA24, not an optional preprocessing step.


### 11.3. Error Analysis (FP, FN, Difficult Cases)

**False positives (77 on test):** relatively rare at precision 0.95. Likely sources include vegetation texture, shadows, or partial non-animal shapes scored above `conf_threshold` 0.25. NMS IoU 0.45 already suppresses duplicate boxes; remaining FP are probably single spurious detections rather than burst duplicates.

**False negatives (306 on test):** the main limitation. Plausible causes on camera trap data:

- **Small or distant animals** — low pixel footprint after resize to 640×640 training size.
- **Night / IR frames** — low contrast, noise, or glare unlike daytime pretraining statistics.
- **Partial occlusion** — animal behind vegetation or at frame edge; box may be tight and hard to match at IoU 0.5.
- **Burst-adjacent difficulty** — even without leakage, some test **events** are inherently harder (motion blur, animal leaving frame) than the near-duplicates that leakage would have made easy.

**FP vs FN asymmetry:** tuning `conf_threshold` downward would trade more FP for higher recall; at 0.25 the model is already conservative (high precision). Improving recall without sacrificing precision would need better capacity, more training signal, or harder-negative mining—not only threshold changes.

**Sample smoke test (Section 10.1):** zero TPs on 4 images is **not** contradictory—it reflects tiny N and different thresholds (YOLO sample `conf` 0.02 still produced no boxes on that test draw). Do not use sample metrics to interpret full-model behavior.

*After baseline eval:* optional side-by-side error galleries (Section 10.5) can show whether FN cases overlap across models or are YOLO-specific.


## 12. Limitations

- **Single-class detection only** — species labels in COCO are ignored; we do not measure per-species performance or confusion between animal types (Section 3).

- **Burst grouping is approximate** — pHash + Hamming threshold groups *visually similar* images, not true event IDs from metadata. Some unrelated images may cluster; some burst frames may split across groups if appearance changes (motion, lighting).

- **Anti-leak split is necessary but not perfect** — group-level holdout is much better than a random image split, yet it depends on hash parameters (`threshold 3`, `max_group_size 50`) chosen from a limited sweep.

- **One detector architecture for final numbers** — YOLOv8n (nano) only; no YOLOs/m/l or custom training ablations. Ultralytics val mAP and our shared-eval mAP use different implementations (Section 11.1).

- **Fixed post-processing** — `conf_threshold`, NMS IoU, and match IoU 0.5 are config defaults; we did not systematically sweep inference thresholds on the no-leak test.

- **Incomplete baseline comparison** — full baseline metrics on the no-leak test were not available when this report was written; we cannot yet quantify the gap between classical sliding window and YOLO on the same split (Section 10.2.1).

- **Leaky vs no-leak YOLO contrast** — Section 10.4 compares **val** (leaky) vs **test** (no leak), not the same split; it illustrates bias direction, not a controlled single-variable experiment.

- **Development sample is not representative** — ~20–60 images with random split; Section 10.1 metrics are pipeline checks only.

- **Dataset scope** — non-human ENA24 subset, Eastern North America camera traps; results may not transfer to other regions, sensors, or species distributions.

- **Evaluation scope** — axis-aligned boxes, one IoU threshold for TP/FP; no COCO-style mAP@[0.5:0.95], no size-stratified or per-location breakdown.

- **Compute constraints** — baseline sliding-window eval scales with image size and window count; full test evaluation is impractical on CPU for quick iteration, which slowed baseline experimentation.


## 13. Possible Improvements and Future Work

**Detection quality (YOLO)**

- Train larger YOLO variants (`yolov8s` / `m`) or more epochs with early stopping on no-leak val.
- Tune `conf_threshold` and NMS on val, then report once on test.
- Multi-scale or higher `imgsz` for small/distant animals (main FN driver in Section 11.3).
- Hard-negative mining or focal loss tweaks if FP rise after lowering confidence.

**Baseline pipeline**

- Complete full-dataset train + test eval; compare fairly with YOLO on Section 10.3.
- Coarser sliding-window grid or learned region proposals to reduce compute.
- Fine-tune ResNet backbone (currently frozen) or try stronger window classifier.

**Data and splits**

- Refine pHash grouping (metadata timestamps if available, or embedding-based clustering).
- Per-location or seasonal stratification in group split for domain-shift analysis.
- Enable `--multi_class` for per-species detection and species-aware metrics.

**Evaluation and reporting**

- mAP@[0.5:0.95], size buckets (small/medium/large), and PR curves.
- Error galleries (Section 10.5) and W&B dashboards for qualitative review.
- Export `evaluate_baseline.py` JSON to mirror YOLO eval artifacts.

**Downstream tasks**

- Two-stage **detect then classify** using ENA24 species labels.
- Temporal models on burst groups (optional; different problem than per-image detection).

**Engineering**

- Single reproducible notebook or script that runs anti-leak split → train → eval with frozen seeds.
- CI smoke test on `ena24_sample` to guard regressions in metrics code.


## 14. Conclusions

We built an end-to-end **single-class animal detection** pipeline on **ENA24-detection**: COCO data loading, a **group-level train/val/test split** to mitigate burst-sequence leakage, a **ResNet + sliding-window baseline**, and a **YOLOv8n** target model, all evaluated with **shared class-agnostic metrics** at IoU 0.5.

The most important methodological result is that **split design matters** on camera trap data. A naive random split inflated YOLO metrics (F1 ~0.96, mAP ~0.94 on a leaky val run) compared to an honest **no-leak test** (F1 **0.89**, mAP **0.84** on 1,732 images). The group-level pHash split should be treated as part of the experimental setup, not an afterthought.

On that no-leak test, YOLOv8n achieves **precision ~0.95** and **recall ~0.84** — practical localization performance for a lightweight model on difficult wild imagery. Errors are dominated by **false negatives** (missed animals), not false positives.

The classical baseline demonstrates the course-style detect-by-classification workflow but **full baseline numbers on the same test split remain pending**; the main quantitative story in this report is YOLO under a corrected evaluation protocol.

The codebase is **reproducible** via JSON configs and CLI scripts (Sections 4, 9). Future work can extend detection to **species classification**, stronger detectors, and richer evaluation (Sections 13)—without changing the core lesson that burst-aware splitting is required for trustworthy metrics on ENA24.
